# Getting Started with Kubernetes

## What's covered

- **What Kubernetes actually is** — a system that schedules containers across a fleet of machines and keeps them running the way you said they should
- **The orchestration problem** — what Docker alone gives you and what it doesn't
- **The cluster at 30,000 feet** — control plane, worker nodes, and how they talk
- **The pod** — Kubernetes' smallest deployable unit, and why it isn't "a container"
- **Why Kubernetes matters** — where it runs in production today
- **Getting a local cluster** — `kind`, `minikube`, Docker Desktop, `kubeadm` VM lab, with a recommended path
- **Your first pod** — `kubectl run` and what actually happened
- **The kubectl architecture** — client, API server, and the declarative model
- **Anatomy of a `kubectl` command** — `kubectl <verb> <resource> [name] [flags]`
- **The core pod lifecycle** — `get`, `describe`, `logs`, `exec`, `delete`
- **Imperative vs declarative** — `kubectl run` versus a YAML manifest and `kubectl apply`
- **Getting help when you forget** — `kubectl --help`, `kubectl explain`, the docs

By the end of this notebook you should be able to bring up a local cluster, run your first pod, inspect it, read its logs, exec into it, delete it, and know what each of those words means. We assume you're comfortable with Docker — images, containers, volumes, `docker run` — and with a Linux shell. We assume nothing about pods, nodes, controllers, or the control plane.

## What is Kubernetes?

If you ask ten people what Kubernetes is, you'll get ten slightly different answers. The cleanest one-line definition:

> **Kubernetes is a system that runs containers across a fleet of machines for you, restarts them when they die, replaces them when a machine dies, and gives them a stable way to find each other.**

That sentence does a lot of work. Let's pick it apart.

**A fleet of machines.** Kubernetes assumes you have more than one server. They might be physical, virtual, on-prem, or in the cloud — it doesn't care. Each machine is called a **node**. One small group of nodes runs Kubernetes itself (the **control plane**); the rest run your containers (**worker nodes**).

**Runs containers for you.** You don't tell Kubernetes "start this container on that server." You tell it "I want three copies of this image running somewhere in the cluster" and Kubernetes figures out where. That decision — which container goes on which node — is called **scheduling**.

**Restarts them when they die.** If a container crashes, Kubernetes notices and starts a new one. If a *machine* dies, Kubernetes notices, marks the containers on it as lost, and starts replacements on the surviving nodes. You declare the desired state; Kubernetes works to maintain it.

**Stable way to find each other.** Containers come and go. Their IP addresses change every time. Kubernetes gives you a **Service** — a stable name and stable IP — that always points at whichever containers are currently alive for a given workload. You talk to the Service; Kubernetes does the DNS and routing.

The name is Greek for "helmsman" — the person steering the ship. The "k8s" abbreviation is just *k*, then eight letters, then *s*. It's also commonly called "kube." All three mean the same thing.

## The orchestration problem — what Docker alone doesn't solve

You finished the `docker` repo, so you can already do a lot. You can build images, run containers, mount volumes, wire up networks, and bring up a small stack with `docker compose`. So what's missing?

The honest answer: as soon as you have *more than one machine* and *more than one application*, you run into a cluster of problems that Docker alone wasn't designed to solve.

**Placement.** You have five nodes and twenty containers to run. Which container goes on which node? You probably want CPU-hungry workloads spread out and small workloads packed tightly. Doing that by hand is a full-time job. Kubernetes' **scheduler** does it automatically.

**Self-healing.** A container crashes at 3 a.m. With Docker alone, it stays crashed until someone notices. Kubernetes notices in seconds and starts a replacement. If a *node* dies, Kubernetes redistributes its containers to the survivors.

**Scaling.** You launched two replicas of your API; traffic doubles; you need ten. Doing that with `docker run` means SSH-ing into machines, picking the right ones, running commands. Kubernetes scales by changing one number — `replicas: 10` — and the cluster converges.

**Rollouts and rollbacks.** Shipping version two of your service should not mean "stop all of v1, then start all of v2." Kubernetes does **rolling updates** — start a v2, wait until it's healthy, stop a v1, repeat. If v2 misbehaves, one command rolls everything back.

**Service discovery.** Container `api-7f9c` and container `api-aa31` are both your API. When the `web` container wants to call "the API," it shouldn't have to know which specific containers exist right now. Kubernetes gives every workload a stable **Service** name like `api`. Cluster DNS does the rest.

**Configuration and secrets.** Twenty services need the same database URL; one of them needs a TLS certificate. With Docker alone you wire env vars and mount files by hand. Kubernetes has first-class **ConfigMaps** and **Secrets** that pods reference declaratively.

Docker still does the actual container work. Kubernetes is what you put *on top* of Docker (or, more precisely, on top of a container runtime — we'll get to that distinction in notebook 08) once you have more containers than you can supervise personally.

## The cluster at 30,000 feet

A Kubernetes cluster is split into two kinds of machines: the **control plane** and the **worker nodes**. They each run a specific set of programs, talk to each other over the network, and together form one logical "cluster."

```
       Control plane                              Worker node
+----------------------------+         +---------------------------+
| +--------+  +-----------+  |         |  +-------------------+    |
| | etcd   |<-|api-server |<-+---------+->|   kubelet         |    |
| +--------+  +-----------+  |  HTTPS  |  +---------+---------+    |
|              ^      ^      |         |            |              |
|              |      |      |         |  +---------v---------+    |
|   +----------+      +----+ |         |  | container runtime |    |
|   |                      | |         |  |   (containerd)    |    |
| +-+--------+  +----------+-+         |  +---------+---------+    |
| | scheduler|  | controller |         |            |              |
| +----------+  | manager    |         |  +---------v---------+    |
|               +------------+         |  |  your containers  |    |
|                                      |  |     (in pods)     |    |
|                                      |  +-------------------+    |
|                                      |  +-------------------+    |
|                                      |  |    kube-proxy     |    |
|                                      |  +-------------------+    |
+--------------------------------------+--+--------------------+---+
```

Don't try to memorise the boxes yet. The shape to take away:

- **`etcd`** — a distributed key-value store. The cluster's source of truth. Every object you create in Kubernetes ends up as a row in `etcd`.
- **`kube-apiserver`** — the front door. Everything that wants to read or change cluster state — your `kubectl`, every other component, every controller — goes through the API server. Nothing talks to `etcd` directly except the API server.
- **`kube-scheduler`** — decides which node a new pod should land on.
- **`kube-controller-manager`** — runs the little background loops that watch the world and make it match desired state. More on controllers shortly.
- **`kubelet`** — runs on every worker node. Talks to the API server, gets told "you need to run these pods," and tells the local container runtime to make it happen.
- **`kube-proxy`** — also on every node. Implements the network plumbing that makes Services work.
- **Container runtime** — what actually executes containers on the node. Usually `containerd` (the same thing you met in the docker repo). The CRI — container runtime interface — is what the kubelet uses to talk to it.

We'll open up each of these properly in notebook 08. For now, just know: **the control plane decides; the nodes execute.**

## The pod — Kubernetes' smallest unit

Here's the first thing that surprises people coming from Docker. Kubernetes' smallest deployable thing is not a container. It's a **pod**.

A **pod is one or more containers that share a network and a storage scope, scheduled together onto the same node, and treated as a single unit by Kubernetes.**

For 95% of pods, there is exactly one container inside. So most of the time, "pod" and "container" feel interchangeable. But the abstraction matters because of the 5%.

**Shared network.** All containers in a pod share an IP address and a port space. They can talk to each other on `localhost`. From outside, the pod has one IP — you can't tell the difference between its containers from across the network.

**Shared storage scope.** Volumes are declared on the pod and mounted into whichever containers need them. So a sidecar container can write to the same directory the main container reads from.

**Same node, same lifecycle.** A pod is scheduled atomically onto one node. All its containers start there, live there, and die there together. If the pod is restarted, all its containers restart.

**Why bother?** The pod abstraction exists for *helper containers* — patterns like:

- **Sidecars** — a small companion container that adds a capability, like log shipping or a service mesh proxy.
- **Init containers** — run-once containers that prepare the pod before the main container starts (e.g. fetch a config file, wait for a database to be ready).
- **Adapter and ambassador patterns** — older names for sidecars that translate or proxy traffic in or out.

If you only have one container, the pod is a thin wrapper. If you have a main process plus one or two helpers that *must* live on the same machine and share a filesystem with it, the pod is the right unit.

In this notebook and the next, every pod we run will have exactly one container — to keep the noise down. Multi-container patterns show up properly from notebook 03 onwards.

## Why Kubernetes matters

Containers in production today mostly mean *Kubernetes* in production. The list of places it runs:

- **The big three managed offerings.** EKS (AWS), GKE (Google Cloud), AKS (Azure). All three give you a cluster with the control plane managed for you; you bring the workloads. Tens of thousands of companies run on these.
- **On-prem and edge.** OpenShift (Red Hat's distribution), Rancher, vanilla `kubeadm` clusters. Banks, telcos, and anything regulated tend to live here.
- **Developer laptops.** `kind`, `minikube`, Docker Desktop's bundled Kubernetes. The cluster you'll spin up later in this notebook.
- **CI/CD.** Tekton, Argo Workflows, and Jenkins X are all Kubernetes-native CI systems. GitHub Actions self-hosted runners often run *as pods*.
- **The platform layer.** Many "internal developer platforms" — Backstage, Spotify-style portals, Heroku-style PaaS — are built on top of Kubernetes underneath.

Two facts to absorb.

**Kubernetes is the default.** When a company says "we run on the cloud" in 2026, the next sentence is usually "...on Kubernetes." Skipping it is increasingly rare.

**The CKA is the certification.** The **Certified Kubernetes Administrator** exam, run by the Linux Foundation and the CNCF, is the most widely-recognised credential. This course is structured to take you from zero through to ready-to-sit. By notebook 10 you'll have hands-on familiarity with every domain the exam tests — install, configuration, workloads, services, networking, storage, security, and troubleshooting.

## Getting a local cluster

You only learn Kubernetes by running pods. You need a cluster. Four good options, ordered roughly by simplicity.

**Option 1 — `kind` (Kubernetes IN Docker).** A single binary that spins up a fresh cluster by running each Kubernetes "node" as a Docker container. Boots in under a minute, single-node by default but supports multi-node, easy to reset.

```bash
# macOS
brew install kind

# Linux
go install sigs.k8s.io/kind@latest   # or download the binary

kind create cluster --name learn      # creates a single-node cluster
kind delete cluster --name learn      # wipes it
```

- Pros: fastest start, multi-node works, easy to script, used heavily in CI.
- Cons: nodes are Docker containers, not VMs — fine for learning, slightly different from real installs.

**Option 2 — `minikube`.** Veteran tool. Runs a single-node cluster inside a VM, a Docker container, or your local container runtime. Has add-ons (`minikube addons enable ingress`) that bundle things you'd otherwise install yourself.

```bash
brew install minikube
minikube start
```

- Pros: very forgiving, batteries-included add-ons.
- Cons: slower to boot than `kind`; single-node by default.

**Option 3 — Docker Desktop's bundled Kubernetes.** If you've already got Docker Desktop, there's a checkbox in settings labelled "Enable Kubernetes." Tick it, wait a few minutes.

- Pros: zero new installs if you have Docker Desktop.
- Cons: not always the latest Kubernetes version; restart-to-reset is heavyweight.

**Option 4 — `kubeadm` on VMs.** The "real" install — bring up three or more Linux VMs, install `kubelet`, `kubeadm`, and `kubectl`, run `kubeadm init` on one, `kubeadm join` on the others. We do this end-to-end in notebook 08.

- Pros: matches production exactly. CKA expects this.
- Cons: heaviest setup; not where you want to start on day one.

**For this notebook and chapters 02–07**, use **`kind`** — it's the lightest path to a working cluster. We'll bring up a `kubeadm` lab in notebook 08 when we need to see real control-plane components.

### Installing `kubectl`

Whichever cluster option you pick, you also need **`kubectl`** — the command-line client that talks to the cluster. `kind` and `minikube` will set it up for you on first run, but the explicit install is:

```bash
# macOS
brew install kubectl

# Linux
curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
sudo install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl
```

### Verify the cluster

Three commands tell you the cluster is healthy:

- `kubectl version` — shows the client and server versions. If you see *both*, `kubectl` can reach the API server.
- `kubectl cluster-info` — prints the URL of the API server and a few add-on endpoints.
- `kubectl get nodes` — lists the nodes. With `kind` you'll see one named `learn-control-plane`; with a multi-node cluster you'd see one row per machine.

In [ ]:
!kubectl version
!echo '---'
!kubectl cluster-info
!echo '---'
!kubectl get nodes

## Your first pod — `kubectl run`

The Kubernetes equivalent of "hello world":

```bash
kubectl run hello --image=nginx:alpine
```

That single command does a surprising amount. The sequence under the hood:

1. `kubectl` parses the command and builds a **Pod object** in memory — a small JSON document that says "I want a pod named `hello` whose only container is `nginx:alpine`."
2. It sends that object to the **API server** as an HTTP `POST`.
3. The API server validates the object and writes it to `etcd`.
4. The **scheduler** notices that there's a pod in `etcd` with no node assigned. It picks a node (we only have one in `kind`) and writes that decision back through the API server.
5. The **kubelet** on the chosen node sees "I have a new pod to run." It tells the container runtime to pull the `nginx:alpine` image and start a container.
6. The container starts. The kubelet updates the pod's status — `Pending` → `ContainerCreating` → `Running` — and that status flows back to `etcd`.
7. When you run `kubectl get pods`, you read the latest status from the API server.

You did the work of step 1 and step 2. Everything else happened on its own.

Notice what `kubectl run` did *not* do. It did not connect to a server and SSH-launch a container. It wrote an object to a database and went home. The rest is a *controller* somewhere watching that database and reacting. This is the heart of Kubernetes — **declarative state plus controllers that reconcile reality to it**. You'll see this same pattern with every other object in this course.

In [ ]:
!kubectl run hello --image=nginx:alpine
!echo '---'
# Watch the pod come up — Pending, ContainerCreating, then Running
!kubectl get pods
!echo '---'
# Same listing with the node and pod IP
!kubectl get pods -o wide

## How `kubectl` actually works

When you typed `kubectl run hello --image=nginx:alpine`, you used the **`kubectl`** client — a single binary on your laptop. But `kubectl` didn't run the pod itself. It built an object and sent it to the API server.

```
+-------------------+    HTTPS REST    +-----------------------+
|  kubectl          | ---------------> |  kube-apiserver       |
|  (your terminal)  |                  |   - authenticates     |
|  reads            |                  |   - authorises        |
|  ~/.kube/config   |                  |   - validates         |
+-------------------+                  |   - writes to etcd    |
                                       +-----------+-----------+
                                                   |
                                                   v
                                       +-----------------------+
                                       |  etcd                 |
                                       |   (cluster state)     |
                                       +-----------------------+
```

A few things to lock in:

- **`kubectl` is stateless.** It reads `~/.kube/config` to find the API server URL and credentials, then makes HTTPS calls. Anyone with that file can drive the cluster.
- **Every component talks to the API server.** The scheduler, the kubelet, the controllers, even `kubectl` — none of them talk to `etcd` directly. The API server is the *only* writer.
- **Authentication and authorisation happen at the API server.** Who you are is figured out from the credential in your kubeconfig; what you're allowed to do is decided by **RBAC** — covered in notebook 10.
- **The watch model.** Components like the scheduler and the kubelet don't *poll* the API server. They open a streaming watch — "tell me whenever a pod object changes" — and react. That's how the cluster stays responsive without hammering the API.

Why does this matter? Two reasons. First, when something goes wrong, the question is almost always "what does the API server think the state is?" and "what does the node actually think the state is?" Bridging those two views is the troubleshooting flow. Second, the API-centric design means *everything* in Kubernetes is just an object you create, read, update, or delete — pods, services, configmaps, secrets, even nodes themselves. Learn one API; learn them all.

## Anatomy of a `kubectl` command

Most `kubectl` commands look like this:

```
kubectl <verb>   <resource>   [name]   [flags]
         |         |            |        |
         |         |            |        +-- e.g. -n my-ns, -o yaml, -l app=web
         |         |            +-- specific object, or omitted for all
         |         +-- pods, deployments, services, nodes, ...
         +-- get, describe, create, apply, delete, logs, exec, edit, ...
```

Examples:

- `kubectl get pods` — list all pods in the current namespace
- `kubectl get pod hello` — show one pod
- `kubectl describe pod hello` — verbose details and recent events for that pod
- `kubectl logs hello` — stdout and stderr of the pod's container
- `kubectl exec -it hello -- sh` — open a shell inside the pod
- `kubectl delete pod hello` — delete the pod
- `kubectl get nodes` — list nodes
- `kubectl get all -n kube-system` — list common resource kinds in the `kube-system` namespace

A few conventions worth memorising now:

- **Resource names are pluralisable.** `pod`, `pods`, and `po` all mean the same thing. `deployment`, `deployments`, `deploy`. Most kinds have a 2–4 letter short alias. `kubectl api-resources` lists them.
- **Namespace flag.** Pods live in namespaces. The default namespace is called `default`. Use `-n <ns>` to target a different one. `-A` (or `--all-namespaces`) means "across all namespaces" for `get`-style commands.
- **Output format.** Append `-o wide` for extra columns, `-o yaml` for the full object, `-o json` for JSON, `-o jsonpath=...` for a single field. Critical for the exam.
- **Aliases.** Many people alias `kubectl` to `k` in their shell. We'll write `kubectl` in full throughout these notebooks to keep things explicit.

## The core pod lifecycle

A pod moves through a small set of **phases**:

```
            kubectl run / kubectl apply
                         |
                         v
                  +-------------+
                  |  Pending    |  scheduled, image pulling, init containers
                  +------+------+
                         |
                         v
                  +-------------+
                  |  Running    |  at least one container is up
                  +------+------+
                         |
              +----------+----------+
              |                     |
              v                     v
      +-------------+         +-------------+
      |  Succeeded  |         |   Failed    |
      |  (exit 0)   |         |  (non-zero) |
      +-------------+         +-------------+

                  +-------------+
                  |  Unknown    |  node lost contact with API server
                  +-------------+
```

The everyday commands for living with a pod:

- **`kubectl get pods`** — list pods in the current namespace. Add `-w` to watch live, `-o wide` for the node and IP, `-A` for all namespaces.
- **`kubectl describe pod <name>`** — verbose details: events, image, IP, conditions, the last reason for any container restart. The single most useful "why is my pod sad" command.
- **`kubectl logs <name>`** — stdout and stderr of the pod's main container. Add `-f` to follow (like `tail -f`), `--previous` for the *previous* container instance after a crash.
- **`kubectl exec -it <name> -- <command>`** — run a command inside the pod's container. The `--` separates `kubectl` flags from the command you want to run. `kubectl exec -it hello -- sh` opens an interactive shell.
- **`kubectl port-forward <name> <local>:<pod>`** — tunnel a local port to a port on the pod. Handy for poking at services that aren't exposed yet.
- **`kubectl delete pod <name>`** — delete the pod. With a bare pod (not managed by a controller), it stays gone. With a pod owned by a Deployment or DaemonSet, a replacement appears within seconds — because the controller is busy reconciling desired state.

Pods can be referred to by **name** in the current namespace, or fully qualified as `<namespace>/<name>` in some contexts. Most commands also accept **label selectors** — `-l app=web` — to operate on every pod matching a label. Labels show up properly in notebook 02.

In [ ]:
# Verbose details and recent events for our pod
!kubectl describe pod hello | head -30
!echo '---'
# The container's logs (nginx's startup output)
!kubectl logs hello | tail -5
!echo '---'
# Run a one-shot command inside the pod
!kubectl exec hello -- nginx -v
!echo '---'
# Delete it
!kubectl delete pod hello
!kubectl get pods

## Imperative vs declarative

`kubectl run hello --image=nginx:alpine` is an **imperative** command — you say *what to do right now*. It's fast for learning and ad-hoc tasks, and it's accepted on the exam under time pressure. But it's not how real systems are managed.

The grown-up way is **declarative**: you write a YAML file that describes the object you want, then ask Kubernetes to make reality match.

```yaml
# pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: hello
  labels:
    app: hello
spec:
  containers:
    - name: nginx
      image: nginx:alpine
      ports:
        - containerPort: 80
```

Apply it with:

```bash
kubectl apply -f pod.yaml
```

Why declarative wins for anything non-throwaway:

- **Version-controllable.** YAML files live in Git. You can diff, review, and roll back.
- **Idempotent.** Run `kubectl apply -f pod.yaml` ten times in a row; you get one pod. `kubectl run` would error on the second attempt.
- **Composable.** A directory of YAML files describes a whole environment. `kubectl apply -f ./manifests/` brings it all up.
- **Reusable.** Tools like Helm and Kustomize template these files for multiple environments.

The bridge between the two worlds — and a trick worth memorising for the CKA exam — is to *generate* YAML from an imperative command:

```bash
kubectl run hello --image=nginx:alpine --dry-run=client -o yaml > pod.yaml
```

`--dry-run=client` means "don't actually send this to the cluster, just build the object." `-o yaml` then prints it. You get a working manifest in one shot, which you can edit and `apply`. Most CKA candidates write their YAML this way — it's faster than typing it from scratch and harder to typo.

Notebook 02 goes deep on the YAML structure — `apiVersion`, `kind`, `metadata`, `spec`. This is your first look at it.

In [ ]:
# Generate a pod manifest without creating anything
!kubectl run hello --image=nginx:alpine --dry-run=client -o yaml

## Getting help — `--help`, `explain`, and the docs

You won't memorise every flag of every command, or every field of every object. You don't need to.

- **`kubectl --help`** — top-level command list.
- **`kubectl <verb> --help`** — flags for a specific verb. `kubectl get --help`, `kubectl run --help`, `kubectl apply --help`.
- **`kubectl explain <resource>`** — describes the fields of an object kind. Pair it with a dotted path to dig in. `kubectl explain pod` shows top-level fields; `kubectl explain pod.spec` shows the spec; `kubectl explain pod.spec.containers` shows the container fields. This is the on-disk version of the API reference and works offline.
- **`kubectl api-resources`** — every resource kind the cluster knows about, with its short name, API group, and whether it's namespaced.

For deeper docs:

- **kubernetes.io/docs** — the canonical reference. The "Concepts" and "Tasks" sections are excellent.
- **The CKA curriculum** is published by the CNCF and lists exactly which docs pages are allowed during the exam. Anything under `kubernetes.io/docs` is fair game.
- **`man kubectl`** on Linux installs.

The flow when you're stuck: `kubectl explain <thing>` for "what fields does this have," `kubectl <verb> --help` for "what flags does this command take," then the docs site for tutorials and conceptual explanations.

In [ ]:
!kubectl --help 2>&1 | head -25
!echo '---'
!kubectl explain pod.spec.containers 2>&1 | head -25

## Cleaning up

A few hours into Kubernetes and you'll have stray pods, leftover deployments, and orphaned configmaps scattered around. Two ways to clean:

- **Targeted** — `kubectl delete pod <name>` for one pod, `kubectl delete -f pod.yaml` to delete whatever a file describes, `kubectl delete <kind> <name>` for any other kind.
- **Bulk** — `kubectl delete <kind> --all` deletes every object of that kind in the current namespace. `kubectl delete all --all -n <ns>` wipes every common kind in a namespace. **Take care** — these are namespaced and they really do delete.

For the cluster itself, when you're done for the day:

```bash
kind delete cluster --name learn        # wipes the kind cluster entirely
# or
minikube delete                          # wipes the minikube cluster
```

The whole cluster comes down in seconds. That throwaway-ness is the point of using `kind` for learning — break it, delete it, recreate it, no consequences.